# EDA kliniczna ISIC 2019

Analiza rozkładu diagnoz oraz związku diagnozy z wiekiem, płcią i lokalizacją anatomiczną. Analiza statystyczna i wykresy dotyczą pięciu najliczniejszych klas: `NV`, `MEL`, `BCC`, `BKL`, `AK`. Testy oceniają zależności w danych, nie dowodzą przyczynowości ani poprawy skuteczności modelu.

Notebook automatycznie pobiera brakujące pliki treningowe CSV z [oficjalnego archiwum ISIC](https://challenge.isic-archive.com/data/#2019): metadane i ground truth (łącznie około 2,5 MB). Obrazy nie są potrzebne. Pliki zapisuje do `data/raw/`. Można podać własne pliki, ustawiając zmienne środowiskowe `ISIC_2019_METADATA` i `ISIC_2019_GROUND_TRUTH`.

In [1]:
from pathlib import Path
import os
import tempfile
from urllib.error import URLError
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency, kruskal

METADATA_PATH = Path(os.environ.get(
    "ISIC_2019_METADATA", "data/raw/ISIC_2019_Training_Metadata.csv"
))
GROUND_TRUTH_PATH = Path(os.environ.get(
    "ISIC_2019_GROUND_TRUTH", "data/raw/ISIC_2019_Training_GroundTruth.csv"
))
FIGURES_DIR = Path("figures/eda")
REPORT_PATH = Path("reports/clinical_eda_statistics.md")
TARGET_CLASSES = ["NV", "MEL", "BCC", "BKL", "AK"]
ALL_ISIC_CLASSES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
OFFICIAL_CSV_URLS = {
    "metadata": "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Metadata.csv",
    "ground_truth": "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv",
}

def download_csv_if_missing(path, url, expected_columns):
    if path.is_file():
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    request = Request(url, headers={"User-Agent": "skin-cancer-umwdm-clinical-eda/1.0"})
    try:
        with urlopen(request, timeout=60) as response:
            content = response.read()
    except (URLError, TimeoutError) as error:
        raise RuntimeError(
            f"Nie udało się pobrać {path.name} z oficjalnego archiwum ISIC. "
            "Sprawdź połączenie internetowe albo ustaw właściwą ścieżkę w zmiennej środowiskowej."
        ) from error

    with tempfile.NamedTemporaryFile(dir=path.parent, suffix=".part", delete=False) as temporary_file:
        temporary_path = Path(temporary_file.name)
        temporary_file.write(content)
    try:
        downloaded_columns = pd.read_csv(temporary_path, nrows=0).columns
        missing_columns = [column for column in expected_columns if column not in downloaded_columns]
        if missing_columns:
            raise ValueError(
                f"Pobrany plik {path.name} nie zawiera oczekiwanych kolumn: {missing_columns}."
            )
        temporary_path.replace(path)
    finally:
        temporary_path.unlink(missing_ok=True)
    print(f"Pobrano z oficjalnego archiwum ISIC: {path}")

sns.set_theme(style="whitegrid", context="notebook")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Metadane: {METADATA_PATH}")
print(f"Ground truth: {GROUND_TRUTH_PATH}")

Metadane: data/raw/ISIC_2019_Training_Metadata.csv
Ground truth: data/raw/ISIC_2019_Training_GroundTruth.csv


## Wczytanie i kontrola danych

Ground truth ISIC 2019 jest kodowany one-hot. Weryfikujemy pojedynczą etykietę na obraz i łączymy pliki po identyfikatorze `image`; w przypadku scalonego pliku etykieta `diagnosis` jest używana bez dodatkowego joinu.

In [2]:
download_csv_if_missing(
    METADATA_PATH,
    OFFICIAL_CSV_URLS["metadata"],
    ["image", "age_approx", "sex", "anatom_site_general"],
)

df = pd.read_csv(METADATA_PATH)
if "diagnosis" not in df.columns:
    if "image" not in df.columns:
        raise ValueError("Metadane muszą zawierać kolumnę 'image' do połączenia z ground truth.")
    download_csv_if_missing(
        GROUND_TRUTH_PATH,
        OFFICIAL_CSV_URLS["ground_truth"],
        ["image", *ALL_ISIC_CLASSES],
    )
    labels = pd.read_csv(GROUND_TRUTH_PATH)
    missing_label_columns = [c for c in ALL_ISIC_CLASSES if c not in labels.columns]
    if "image" not in labels.columns or missing_label_columns:
        raise ValueError(
            "Ground truth musi zawierać kolumnę 'image' oraz kolumny one-hot "
            f"{ALL_ISIC_CLASSES}; brakujące: {missing_label_columns}."
        )
    if labels["image"].duplicated().any():
        raise ValueError("Identyfikator 'image' nie jest unikalny w ground truth.")
    one_hot = labels[ALL_ISIC_CLASSES].apply(pd.to_numeric, errors="coerce")
    if one_hot.isna().any().any() or not one_hot.sum(axis=1).eq(1).all():
        raise ValueError("Każdy wiersz ground truth musi mieć dokładnie jedną etykietę one-hot.")
    labels = labels[["image"]].assign(diagnosis=one_hot.idxmax(axis=1))
    df = df.merge(labels, on="image", how="left", validate="one_to_one")

required_columns = ["diagnosis", "age_approx", "sex", "anatom_site_general"]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Brak wymaganych kolumn w danych: {missing_columns}")

df["diagnosis"] = df["diagnosis"].astype("string").str.strip().str.upper()
df["age_approx"] = pd.to_numeric(df["age_approx"], errors="coerce")
for column in ["sex", "anatom_site_general"]:
    df[column] = df[column].astype("string").str.strip()
    df.loc[df[column].eq(""), column] = pd.NA

unknown_classes = sorted(set(df["diagnosis"].dropna()) - set(ALL_ISIC_CLASSES))
if unknown_classes:
    raise ValueError(f"Nieznane diagnozy spoza klas ISIC 2019: {unknown_classes}")
if df["diagnosis"].isna().any():
    raise ValueError("Znaleziono rekordy bez diagnozy; sprawdź kompletność ground truth.")

class_counts = df["diagnosis"].value_counts().reindex(ALL_ISIC_CLASSES, fill_value=0)
class_order = sorted(
    (c for c in TARGET_CLASSES if class_counts[c] > 0),
    key=lambda c: (-int(class_counts[c]), TARGET_CLASSES.index(c)),
)
if len(class_order) < 2:
    raise ValueError("Do porównań potrzeba danych z co najmniej dwóch klas spośród NV, MEL, BCC, BKL, AK.")

analysis_df = df.loc[df["diagnosis"].isin(class_order)].copy()
print(f"Wczytane rekordy: {len(df):,}; rekordy w pięciu analizowanych klasach: {len(analysis_df):,}")
display(
    class_counts[class_counts > 0]
    .sort_values(ascending=False)
    .rename("liczba")
    .to_frame()
)

Wczytane rekordy: 25,331; rekordy w pięciu analizowanych klasach: 24,211


,liczba
diagnosis,
NV,12875
MEL,4522
BCC,3323
BKL,2624
AK,867
SCC,628
VASC,253
DF,239


## Rozkład klas i braków danych

Pierwszy wykres pokazuje liczebność oraz udział każdej z pięciu najliczniejszych klas w całym zbiorze. Pozostałe klasy pozostają w danych wejściowych, ale są wyłączone z tej analizy porównawczej.

In [3]:
class_summary = (
    analysis_df["diagnosis"].value_counts()
    .reindex(class_order)
    .rename("count")
    .rename_axis("diagnosis")
    .reset_index()
)
class_summary["percent_of_dataset"] = 100 * class_summary["count"] / len(df)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=class_summary, x="diagnosis", y="count", order=class_order, color="#4472C4", ax=ax)
for index, row in class_summary.iterrows():
    ax.text(index, row["count"], f"{row['count']:,}\n{row['percent_of_dataset']:.1f}%",
            ha="center", va="bottom", fontsize=9)
ax.set(title="Liczebność pięciu najczęstszych klas ISIC 2019", xlabel="Diagnoza", ylabel="Liczba przypadków")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "class_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

missingness = (
    df[required_columns].isna().mean().mul(100)
    .rename("Braki danych (%)")
    .to_frame()
    .rename_axis("Zmienna")
)
display(missingness.round(2))
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.heatmap(missingness, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0, vmax=100,
            cbar_kws={"label": "% braków"}, ax=ax)
ax.set(title="Odsetek brakujących wartości", xlabel="", ylabel="")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "missingness.png", dpi=200, bbox_inches="tight")
plt.show()

/var/folders/dc/_pk4n_xx4f1b737kr5jqgb5r0000gn/T/ipykernel_89810/2866318642.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Braki danych (%)
Zmienna,
diagnosis,0.00
age_approx,1.73
sex,1.52
anatom_site_general,10.39


/var/folders/dc/_pk4n_xx4f1b737kr5jqgb5r0000gn/T/ipykernel_89810/2866318642.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Wiek oraz lokalizacja anatomiczna

Violinplot pokazuje rozkład wieku w każdej klasie. Heatmapa lokalizacji jest normalizowana w obrębie diagnozy: każda kolumna sumuje się do 1 i przedstawia `P(lokalizacja | diagnoza)`. Brak lokalizacji jest ujęty jako `Unknown`.

In [4]:
age_data = analysis_df.dropna(subset=["age_approx"])
fig, ax = plt.subplots(figsize=(9, 5))
sns.violinplot(data=age_data, x="diagnosis", y="age_approx", order=class_order,
               inner="quartile", cut=0, color="#80B1D3", ax=ax)
ax.set(title="Rozkład wieku według diagnozy", xlabel="Diagnoza", ylabel="Wiek (lata)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "age_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

site_data = analysis_df.assign(
    anatom_site_general=analysis_df["anatom_site_general"].fillna("Unknown")
)
site_counts = pd.crosstab(site_data["anatom_site_general"], site_data["diagnosis"])
site_counts = site_counts.reindex(columns=class_order, fill_value=0)
site_normalized = site_counts.div(site_counts.sum(axis=0), axis=1)
fig, ax = plt.subplots(figsize=(10, max(5, 0.38 * len(site_normalized))))
sns.heatmap(site_normalized, annot=True, fmt=".1%", cmap="Blues",
            cbar_kws={"label": "Udział w obrębie diagnozy"}, ax=ax)
ax.set(title="Lokalizacja anatomiczna w obrębie diagnozy (kolumny = 100%)",
       xlabel="Diagnoza", ylabel="Lokalizacja anatomiczna")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "site_diagnosis_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

/var/folders/dc/_pk4n_xx4f1b737kr5jqgb5r0000gn/T/ipykernel_89810/3156933082.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/dc/_pk4n_xx4f1b737kr5jqgb5r0000gn/T/ipykernel_89810/3156933082.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Testy statystyczne

Testy chi-kwadrat badają niezależność diagnozy i (oddzielnie) lokalizacji oraz płci. Test Kruskala-Wallisa porównuje wiek między klasami, nie zakładając normalności rozkładu. Wartości p nie mierzą siły ani wartości predykcyjnej cechy; przy licznych małych wartościach oczekiwanych przybliżenie chi-kwadrat może być niedokładne.

In [5]:
def chi_square_result(data, feature):
    table_data = data[[feature, "diagnosis"]].dropna().copy()
    table = pd.crosstab(table_data[feature], table_data["diagnosis"])
    table = table.reindex(columns=class_order, fill_value=0)
    table = table.loc[table.sum(axis=1) > 0, table.sum(axis=0) > 0]
    if table.shape[0] < 2 or table.shape[1] < 2:
        raise ValueError(f"Za mało kategorii w tabeli '{feature}' do testu chi-kwadrat.")
    chi2, p_value, degrees_of_freedom, expected = chi2_contingency(table, correction=False)
    cramer_v = np.sqrt(
        chi2 / (table.to_numpy().sum() * min(table.shape[0] - 1, table.shape[1] - 1))
    )
    return {
        "feature": feature,
        "chi2": float(chi2),
        "p_value": float(p_value),
        "degrees_of_freedom": int(degrees_of_freedom),
        "n": int(table.to_numpy().sum()),
        "min_expected": float(expected.min()),
        "expected_below_5_percent": float((expected < 5).mean() * 100),
        "cramers_v": float(cramer_v),
    }

chi_results = [
    chi_square_result(analysis_df.assign(
        anatom_site_general=analysis_df["anatom_site_general"].fillna("Unknown")
    ), "anatom_site_general"),
    chi_square_result(analysis_df, "sex"),
]

age_groups = [
    group["age_approx"].dropna().to_numpy()
    for diagnosis in class_order
    if len(group := analysis_df.loc[analysis_df["diagnosis"].eq(diagnosis)])
    and group["age_approx"].notna().any()
]
if len(age_groups) < 2:
    raise ValueError("Do testu Kruskala-Wallisa potrzeba wieku w co najmniej dwóch klasach.")
kw_h, kw_p = kruskal(*age_groups)
kw_epsilon_squared = max(
    0.0, (kw_h - len(age_groups) + 1) / (sum(len(group) for group in age_groups) - len(age_groups))
)
kw_result = {
    "h": float(kw_h),
    "p_value": float(kw_p),
    "epsilon_squared": float(kw_epsilon_squared),
    "degrees_of_freedom": len(age_groups) - 1,
    "n_by_class": {
        diagnosis: int(analysis_df.loc[analysis_df["diagnosis"].eq(diagnosis), "age_approx"].notna().sum())
        for diagnosis in class_order
    },
    "n": int(sum(len(group) for group in age_groups)),
}

display(pd.DataFrame(chi_results).set_index("feature"))
display(pd.Series(kw_result, name="Kruskal-Wallis"))

,chi2,p_value,degrees_of_freedom,n,min_expected,expected_below_5_percent,cramers_v
feature,,,,,,,
anatom_site_general,5540.810071,0.000000e+00,32,24211,1.933749,4.444444,0.239194
sex,159.249655,2.117609e-33,4,23827,406.707685,0.000000,0.081753


h                                                           6572.429495
p_value                                                             0.0
epsilon_squared                                                0.276344
degrees_of_freedom                                                    4
n_by_class            {'NV': 12558, 'MEL': 4437, 'BCC': 3317, 'BKL':...
n                                                                 23774
Name: Kruskal-Wallis, dtype: object

## Eksport wyników

Wyniki są zapisywane jako dwa wykresy rozkładu, heatmapa, wykres braków danych oraz raport Markdown z liczebnościami, brakami i pełnymi statystykami testów.

In [6]:
def format_p(value):
    return "<1e-300" if value < 1e-300 else f"{value:.6g}"

lines = [
    "# Wyniki EDA klinicznej ISIC 2019",
    "",
    "- Źródło: [ISIC 2019](https://challenge.isic-archive.com/data/#2019), treningowe metadane i ground truth (licencja CC BY-NC 4.0).",
    f"- Rekordy wczytane: **{len(df):,}**.",
    f"- Rekordy w pięciu analizowanych klasach: **{len(analysis_df):,}**.",
    f"- Klasy: **{', '.join(class_order)}** (wybrane jako pięć najliczniejszych klas docelowych).",
    "",
    "## Liczebność klas",
    "",
    "| Diagnoza | Liczba | Udział całego zbioru |",
    "|---|---:|---:|",
]
for _, row in class_summary.iterrows():
    lines.append(f"| {row['diagnosis']} | {row['count']:,} | {row['percent_of_dataset']:.2f}% |")

lines.extend(["", "## Braki danych", "", "| Zmienna | Braki (%) |", "|---|---:|"])
for variable, row in missingness.iterrows():
    lines.append(f"| {variable} | {row['Braki danych (%)']:.2f}% |")

lines.extend([
    "", "## Testy chi-kwadrat niezależności", "",
    "| Cecha | χ² | df | p-value | V Craméra | N | Min. oczekiwana | Oczekiwane < 5 (%) |",
    "|---|---:|---:|---:|---:|---:|---:|---:|",
])
for result in chi_results:
    lines.append(
        f"| {result['feature']} | {result['chi2']:.4f} | {result['degrees_of_freedom']} "
        f"| {format_p(result['p_value'])} | {result['cramers_v']:.3f} "
        f"| {result['n']:,} | {result['min_expected']:.3f} "
        f"| {result['expected_below_5_percent']:.1f}% |"
    )
lines.extend([
    "", "## Test Kruskala-Wallisa dla wieku", "",
    f"- H = **{kw_result['h']:.4f}**, df = **{kw_result['degrees_of_freedom']}**, "
    f"p-value = **{format_p(kw_result['p_value'])}**, N = **{kw_result['n']:,}**.",
    f"- Wielkość efektu (ε²) = **{kw_result['epsilon_squared']:.3f}**.",
    "- Liczba kompletnych obserwacji według klasy: "
    + "; ".join(f"{key}: {value:,}" for key, value in kw_result["n_by_class"].items()) + ".",
    "", "## Interpretacja i ograniczenia", "",
    "Testy chi-kwadrat oceniają zależność między diagnozą a płcią/lokalizacją; V Craméra i ε² "
    "są miarami wielkości obserwowanych zależności/różnic. Dla chi-kwadrat przybliżenie jest "
    "zwyczajowo uznawane za wiarygodne, gdy mniej niż 20% komórek ma liczebność oczekiwaną < 5 "
    "i żadna nie ma liczebności < 1; raport podaje te wartości, aby umożliwić kontrolę warunku. "
    "Wielkości efektu są opisowe, nie dowodzą skuteczności predykcyjnej. Istotność statystyczna nie "
    "dowodzi przyczynowości ani wartości predykcyjnej cechy; przy dużym zbiorze nawet niewielkie "
    "różnice mogą dawać małe p-value. Przydatność cech w fuzji multimodalnej należy następnie "
    "ocenić w walidacji modelu bez przecieku danych.",
    "", "## Wygenerowane wykresy", "",
    "- `figures/eda/class_distribution.png`",
    "- `figures/eda/age_distribution.png`",
    "- `figures/eda/site_diagnosis_heatmap.png`",
    "- `figures/eda/missingness.png`",
])
REPORT_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Zapisano raport: {REPORT_PATH}")
print(f"Wykresy zapisano w: {FIGURES_DIR.resolve()}")

Zapisano raport: reports/clinical_eda_statistics.md
Wykresy zapisano w: /Users/julitaskorzycka/Desktop/SEMESTR II/raki/skin-cancer-umwdm-1/figures/eda
